### CNN Preprocessing + fitting with optuna

#### 0. Initialization

In [1]:
import numpy as np
import os
import pandas as pd

from pathlib import Path
from sklearn.model_selection import train_test_split
import torch

from ml.core.config import settings
from ml.core.kernel import SciKitCNN

genres = settings.GENRE_LABELS

#### 1. Preprocessing
Can be skipped if it's done already and the results are saved 

In [ ]:
import librosa
import soundfile as sf
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm

In [ ]:
# download from https://drive.google.com/file/d/1K2LgkkprXLTQB70VKMZlmKCfTSd-ina_/view?usp=drive_link
# and unpack into the rooth directory of the project

data_dir = Path('./data/genres_original/')
data = []

for genre in os.listdir(data_dir):
  genre_path = os.path.join(data_dir,genre)
  for file in os.listdir(genre_path):
    if file.endswith('.wav'):
      file_path = os.path.join(genre_path, file)
      data.append((genre, file_path))
      
df = pd.DataFrame(data,columns = ['genre','file_path']) 

In [ ]:
# splitting to subsets before segmenting to avoid data leakage

train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['genre'])
val_df, test_df = train_test_split(test_df, test_size=2/3, random_state=42, stratify=test_df['genre'])

print(f'Train files: {len(train_df)}')
print(f'Test files: {len(test_df)}')
print(f'Val files: {len(val_df)}')

In [ ]:
segment_length = 4  # seconds
overlap = 2         # seconds
sr = 22050          # sample rate (same for all)

# a function to divide a track into segments with segment_length time
def create_segments(metadata_df):
    segmented = []
    for _, row in tqdm(metadata_df.iterrows(), total=len(metadata_df)):
        file_path = row['file_path']
        genre = row['genre']

        try:
            #audio, file_sr = sf.read(file_path)
            
            audio, file_sr = librosa.load(file_path, sr=settings.SAMPLE_RATE, mono=True,
                                            duration=settings.MAX_AUDIO_LENGTH_SEC,)
            
            if len(audio.shape) > 1:
                audio = np.mean(audio, axis=1)

            if file_sr != sr:
                audio = librosa.resample(audio,orig_sr=file_sr,target_sr=sr)

            segment_samples = segment_length * sr
            step = (segment_length - overlap) * sr

            for start in range(0, len(audio) - segment_samples + 1, step):
                end = start + segment_samples
                segment = audio[start:end]
                segmented.append({'audio': segment,'genre': genre,'source_file': file_path})

        except Exception as e:
            print(f'Error processing {file_path}: {e}')

    return pd.DataFrame(segmented)


In [ ]:
train_segments = create_segments(train_df)
test_segments = create_segments(test_df)
val_segments = create_segments(val_df)

print(f'Train segments: {len(train_segments)}')
print(f'Test segments: {len(test_segments)}')
print(f'Val segments: {len(val_segments)}')

In [ ]:
n_fft = 2048
hop_length = 512
n_mels = 150
image_size = 150

# a function to create spectrograms, based on which a model will be fitted
def extract_mel_spectrograms(segmented_df):
    mel_specs = []
    labels = []
    for _, row in tqdm(segmented_df.iterrows(), total=len(segmented_df)):
        signal = row['audio'] 
        
        mel_spec = librosa.feature.melspectrogram(y=signal,sr=sr,n_fft=n_fft,
                                                  hop_length=hop_length,n_mels=n_mels)
        mel_spec_db = librosa.power_to_db(mel_spec,ref=np.max)
        #min_db = -80.0  # стандартный порог тишины
        #mel_spec_norm = np.clip((mel_spec_db - min_db) / (-min_db), 0, 1)
        mel_tensor = torch.tensor(mel_spec_db,dtype=torch.float32).unsqueeze(0).unsqueeze(0)
        mel_resized = F.interpolate(mel_tensor,size=(image_size, image_size),
                                    mode='bilinear',align_corners=False)
        mel_specs.append(mel_resized.squeeze().numpy())

        labels.append(row['genre'])

    return np.array(mel_specs), np.array(labels)


In [ ]:
def apply_spec_augment(spec, num_masks=2, freq_mask_max=15, time_mask_max=20):
    augmented = spec.copy()
    
    # Frequency Masking 
    for _ in range(num_masks):
        f = np.random.randint(0, freq_mask_max)
        f0 = np.random.randint(0, augmented.shape[0] - f)
        augmented[f0:f0+f, :] = 0.0 
        
    # Time Masking
    for _ in range(num_masks):
        t = np.random.randint(0, time_mask_max)
        t0 = np.random.randint(0, augmented.shape[1] - t)
        augmented[:, t0:t0+t] = 0.0 
        
    return augmented

In [ ]:
X_train, y_train_raw = extract_mel_spectrograms(train_segments)
X_test, y_test_raw = extract_mel_spectrograms(test_segments)
X_val, y_val_raw = extract_mel_spectrograms(val_segments)

# Добавляем аугментированные данные в тренировочную выборку
# Показало не очень хорошие результаты
# X_train_augmented = np.array([apply_spec_augment(img) for img in X_train])
# X_train = X_train_augmented
#X_train = np.concatenate((X_train, X_train_augmented), axis=0)
#y_train_raw = np.concatenate((y_train_raw, y_train_raw), axis=0)

print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

In [ ]:
# Encode categorical labels into one hot

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train_raw)
y_test_encoded = label_encoder.transform(y_test_raw)
y_val_encoded = label_encoder.transform(y_val_raw)

num_classes = len(label_encoder.classes_)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
X_test = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
X_val = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)

y_train = torch.tensor(y_train_encoded, dtype=torch.long)
y_test = torch.tensor(y_test_encoded, dtype=torch.long)
y_val = torch.tensor(y_val_encoded, dtype=torch.long)

y_train = F.one_hot(y_train,num_classes=num_classes).float()
y_test = F.one_hot(y_test,num_classes=num_classes).float()
y_val = F.one_hot(y_val,num_classes=num_classes).float()


In [ ]:
# Save tensors for train / val / test splits

save_dir = Path('./data/saved_not_normed_tensors')
save_dir.mkdir(parents=True, exist_ok=True)

torch.save({'X': X_train, 'y': y_train}, save_dir / 'train_tensors.pt')
torch.save({'X': X_val, 'y': y_val}, save_dir / 'val_tensors.pt')
torch.save({'X': X_test, 'y': y_test}, save_dir / 'test_tensors.pt')

print(f'Saved train tensors to {save_dir / "train_tensors.pt"}')
print(f'Saved val tensors to {save_dir / "val_tensors.pt"}')
print(f'Saved test tensors to {save_dir / "test_tensors.pt"}')

image_size = 150

#### 1.5 Loading data

In [ ]:
# Load tensors before model training

load_dir = Path('./data/saved_tensors')

train_data = torch.load(load_dir / 'train_tensors.pt', map_location='cpu')
val_data = torch.load(load_dir / 'val_tensors.pt', map_location='cpu')
test_data = torch.load(load_dir / 'test_tensors.pt', map_location='cpu')

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val = val_data['X'], val_data['y']
X_test, y_test = test_data['X'], test_data['y']

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

#### 2. Model training
Don't try without GPU, since it'll run for hours in that case

Params below provide current optimal result

In [ ]:
lr = 1e-4
weight_decay=2e-4
batch_size = 32
epoch_n = 50
num_classes = 10
channels = (64, 128, 256)
dropout=0.25
image_size = 150

model = SciKitCNN(channels=channels,num_classes=num_classes,epoch_n=epoch_n, weight_decay=weight_decay,
					  batch_size=batch_size,lr=lr,image_size=image_size, dropout=dropout)

In [ ]:
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

y_pred = model.predict(X_test)

y_test_np = y_test.argmax(dim=1).numpy()

print('Testing accuracy:', accuracy_score(y_test_np, y_pred))
print('Testing roc-auc :', f1_score(y_test_np, y_pred, average="macro"))
print(classification_report(y_test_np,y_pred,target_names=genres))


In [ ]:
filename = Path("./ml/models/model4.pt")
model.save_model(filename)

In [ ]:
filename = Path("./ml/models/model2.pt")

model1 = SciKitCNN()
model1.load_model(filename)

In [ ]:
from ml.utils.utils import preprocess_audio

audio_path = Path("./ml/audio_samples/Dance!.mp3")
X_for_inference = preprocess_audio(audio_path)

model.predict_genre(X_for_inference)

#### 3. Optuna optimization
To change param bounds for optimization go to .ml.utils.utils.Objective

In [ ]:
from ml.utils.utils import ModelOptimization

# n_startup_trials + n_trials - n times to fit a model and evaluate it

model_list = ["CNN"]

OS = ModelOptimization(model_list)
OS.fit(X_train, y_train, X_val, y_val, n_trials=15, n_startup_trials=5)

In [ ]:
from ml.utils.utils import preprocess_audio

audio_path = Path("./ml/audio_samples/Dance!.mp3")
X_for_inference = preprocess_audio(audio_path)

model.predict_genre(X_for_inference)

[{'label': 'disco', 'score': 0.5122389197349548},
 {'label': 'hip-hop', 'score': 0.08596889674663544},
 {'label': 'rock', 'score': 0.07801486551761627},
 {'label': 'classical', 'score': 0.06198340281844139},
 {'label': 'jazz', 'score': 0.057548873126506805},
 {'label': 'metal', 'score': 0.050546519458293915},
 {'label': 'blues', 'score': 0.04598009213805199},
 {'label': 'country', 'score': 0.045119062066078186},
 {'label': 'reggae', 'score': 0.040671080350875854},
 {'label': 'pop', 'score': 0.021928338333964348}]

In [ ]:
audio_path = Path("./ml/audio_samples/String Theocracy (inst).mp3")
X_for_inference = preprocess_audio(audio_path)

model.predict_genre(X_for_inference)

[{'label': 'jazz', 'score': 0.3859366178512573},
 {'label': 'classical', 'score': 0.12593358755111694},
 {'label': 'rock', 'score': 0.10231783986091614},
 {'label': 'hip-hop', 'score': 0.09201157093048096},
 {'label': 'blues', 'score': 0.06995252519845963},
 {'label': 'disco', 'score': 0.0599241629242897},
 {'label': 'country', 'score': 0.057872213423252106},
 {'label': 'metal', 'score': 0.03986167162656784},
 {'label': 'reggae', 'score': 0.0375421866774559},
 {'label': 'pop', 'score': 0.028647735714912415}]

In [ ]:
audio_path = Path("./ml/audio_samples/ST.mp3")
X_for_inference = preprocess_audio(audio_path)

model.predict_genre(X_for_inference)

[{'label': 'jazz', 'score': 0.3126189708709717},
 {'label': 'rock', 'score': 0.10426648706197739},
 {'label': 'hip-hop', 'score': 0.10422510653734207},
 {'label': 'blues', 'score': 0.08631822466850281},
 {'label': 'classical', 'score': 0.08412826061248779},
 {'label': 'country', 'score': 0.07885967195034027},
 {'label': 'pop', 'score': 0.06726687401533127},
 {'label': 'disco', 'score': 0.06677050143480301},
 {'label': 'metal', 'score': 0.050495315343141556},
 {'label': 'reggae', 'score': 0.045050591230392456}]

In [ ]:
audio_path = Path("./ml/audio_samples/Metal.mp3")
X_for_inference = preprocess_audio(audio_path)

model.predict_genre(X_for_inference)

[{'label': 'classical', 'score': 0.2998819053173065},
 {'label': 'metal', 'score': 0.22205844521522522},
 {'label': 'jazz', 'score': 0.11479351669549942},
 {'label': 'rock', 'score': 0.0807783380150795},
 {'label': 'disco', 'score': 0.08063840121030807},
 {'label': 'hip-hop', 'score': 0.07827907800674438},
 {'label': 'country', 'score': 0.04451163858175278},
 {'label': 'reggae', 'score': 0.03886063024401665},
 {'label': 'blues', 'score': 0.02463560365140438},
 {'label': 'pop', 'score': 0.015562509186565876}]